## Cleaning the Data

Load data:

In [1]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/HW GROUP - ADSP 31017 Machine Learning I/Project/Africa_aggregated_data_up_to-2026-02-14.xlsx'

df = pd.read_excel(file_path)

print(df.info())
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265548 entries, 0 to 265547
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   WEEK                 265548 non-null  datetime64[ns]
 1   REGION               265548 non-null  object        
 2   COUNTRY              265548 non-null  object        
 3   ADMIN1               265548 non-null  object        
 4   EVENT_TYPE           265548 non-null  object        
 5   SUB_EVENT_TYPE       265548 non-null  object        
 6   EVENTS               265548 non-null  int64         
 7   FATALITIES           265548 non-null  int64         
 8   POPULATION_EXPOSURE  170127 non-null  float64       
 9   DISORDER_TYPE        265548 non-null  object        
 10  ID                   265543 non-null  float64       
 11  C

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
0,2004-10-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388
1,2005-04-23,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,0,NaN,Political violence,47.0,26.4839,-1.388
2,2005-06-25,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,14,NaN,Political violence,47.0,26.4839,-1.388
3,2008-12-13,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,3,NaN,Political violence,47.0,26.4839,-1.388
4,2009-04-18,Northern Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388


Fill in empty cells:
- If an event was recorded but the event_type is blank or not in the list it is assigned 1 (Strategic Development) because something happened, but it wasn't a violent war.
- If there's nothing that week for a country, it is a "Peaceful Week". We add the week to the data and add a 0.

In [2]:
severity_map = {
    'Strategic developments': 1,
    'Protests': 2,
    'Riots': 3,
    'Violence against civilians': 4,
    'Explosions/Remote violence': 5,
    'Battles': 6
}
df['severity_score'] = df['EVENT_TYPE'].map(severity_map).fillna(1)

In [3]:
# Aggregate counts and find the max severity reached in that week
aggr_df = df.groupby(['COUNTRY', 'REGION', 'WEEK']).agg(
    weekly_events=('EVENTS', 'sum'),
    max_severity=('severity_score', 'max'),
    total_fatalities=('FATALITIES', 'sum')
).reset_index()

# Sort by country and time to ensure lags are calculated correctly
aggr_df = aggr_df.sort_values(['COUNTRY', 'WEEK'])

In [4]:
# 1. Indicator: Was there conflict in the previous week?
# We check if the 'weekly_events' from the previous row for the SAME country is > 0
aggr_df['prev_week_events'] = aggr_df.groupby('COUNTRY')['weekly_events'].shift(1).fillna(0)
aggr_df['has_prior_conflict'] = (aggr_df['prev_week_events'] > 0).astype(int)

# 2. Indicator: Did conflict escalate?
# Compare current max severity to previous max severity
aggr_df['prev_severity'] = aggr_df.groupby('COUNTRY')['max_severity'].shift(1).fillna(0)
aggr_df['is_escalated'] = (aggr_df['max_severity'] > aggr_df['prev_severity']).astype(int)

# 3. Categorical Indicators for TensorFlow (Country and Region)
aggr_df['country_code'] = aggr_df['COUNTRY'].astype('category').cat.codes
aggr_df['region_code'] = aggr_df['REGION'].astype('category').cat.codes

In [5]:
# Create a complete grid of all Countries and all Weeks
all_weeks = aggr_df['WEEK'].unique()
all_countries = aggr_df['COUNTRY'].unique()
grid = pd.MultiIndex.from_product([all_countries, all_weeks], names=['COUNTRY', 'WEEK']).to_frame(index=False)

# Merge our aggregated data back onto this grid
final_df = pd.merge(grid, aggr_df, on=['COUNTRY', 'WEEK'], how='left').fillna(0)

Check:

In [7]:
raw_total = df['EVENTS'].sum()
clean_total = final_df['weekly_events'].sum()

print(f"Original Events: {raw_total}")
print(f"Aggregated Events: {clean_total}")

if raw_total == clean_total:
    print("No events were lost during aggregation.")
else:
    print("Sums do not match.")

Original Events: 475008
Aggregated Events: 475008.0
No events were lost during aggregation.


In [10]:
test_country = all_countries[0] # This picks the first country in your list (e.g., Algeria)
country_data = final_df[final_df['COUNTRY'] == test_country]

print(f"Checking data for: {test_country}")
check_cols = ['WEEK', 'weekly_events', 'max_severity', 'prev_severity', 'is_escalated', 'has_prior_conflict']
display(country_data[check_cols].head(15))

Checking data for: Algeria


,WEEK,weekly_events,max_severity,prev_severity,is_escalated,has_prior_conflict
0,1996-12-28,3.0,4.0,0.0,1.0,0.0
1,1997-01-04,9.0,6.0,4.0,1.0,1.0
2,1997-01-11,10.0,6.0,6.0,0.0,1.0
3,1997-01-18,13.0,6.0,6.0,0.0,1.0
4,1997-01-25,7.0,6.0,6.0,0.0,1.0
5,1997-02-01,3.0,4.0,6.0,0.0,1.0
6,1997-02-08,1.0,4.0,4.0,0.0,1.0
7,1997-02-15,1.0,4.0,4.0,0.0,1.0
8,1997-02-22,3.0,4.0,4.0,0.0,1.0
9,1997-03-29,3.0,4.0,4.0,0.0,1.0


In [9]:
# Check for sparsity
sparsity = (final_df['weekly_events'] == 0).mean()
print(f"Matrix Sparsity: {sparsity:.2%}")

Matrix Sparsity: 56.70%


Integer indices, so TensorFlow works

In [12]:
# 1. Create unique mapping for Countries and Weeks
final_df['country_id'] = final_df['COUNTRY'].astype('category').cat.codes
final_df['week_id'] = final_df['WEEK'].astype('category').cat.codes

# 2. Extract the total counts for the Embedding layers
num_countries = final_df['country_id'].nunique()
num_weeks = final_df['week_id'].nunique()

print(f"Ready for TensorFlow:")
print(f"- Unique Countries (Rows): {num_countries}")
print(f"- Unique Weeks (Columns): {num_weeks}")
print(f"- Total Training Samples: {len(final_df)}")

# 3. Save this to a CSV
final_df.to_csv('Africa_Conflict_Cleaned_ML_Ready.csv', index=False)

Ready for TensorFlow:
- Unique Countries (Rows): 58
- Unique Weeks (Columns): 1521
- Total Training Samples: 88218
